# Sulama İhtiyacını Tahmin Etmek

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import balanced_accuracy_score
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

In [2]:
train= pd.read_csv('train(1).csv')
test= pd.read_csv('test(1).csv')

In [3]:
train.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [4]:
test.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [5]:
# Hedef Değişkeni Sayısallaştırma (Low:0, Medium:1, High:2)
# LightGBM metin etiketleri doğrudan hedef değişken olarak kabul etmeyebilir
target_mapper = {'Low': 0, 'Medium': 1, 'High': 2}
inv_target_mapper = {0: 'Low', 1: 'Medium', 2: 'High'}
train['Irrigation_Need'] = train['Irrigation_Need'].map(target_mapper)

In [6]:
# Kategorik (Metin) Sütunları Güvenli Şekilde Sayısallaştırma
kategorik_sutunlar = train.select_dtypes(include=['object']).columns

In [7]:
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train[kategorik_sutunlar] = encoder.fit_transform(train[kategorik_sutunlar].astype(str))
test[kategorik_sutunlar] = encoder.transform(test[kategorik_sutunlar].astype(str))


In [8]:
# Tarımsal Alan İçin Özellik Mühendisliği (Feature Engineering)
for df in [train, test]:
    # Su İndeksi: Yağan yağmur ile önceki sulamanın toplamı
    df['total_water_received'] = df['Rainfall_mm'] + df['Previous_Irrigation_mm']
    # Kuruluk / Buharlaşma Riski: Sıcaklık arttıkça ve nem düştükçe su ihtiyacı artar
    df['evaporation_risk'] = df['Temperature_C'] / (df['Humidity'] + 1e-5)
    # Güneşlenme verimliliği
    df['sunlight_moisture_ratio'] = df['Sunlight_Hours'] / (df['Soil_Moisture'] + 1e-5)

print("2. Özellik mühendisliği tamamlandı. Modelleme başlıyor...")


2. Özellik mühendisliği tamamlandı. Modelleme başlıyor...


In [9]:
# Girdileri (X) ve Hedefi (y) Ayırma
giris_sutunlari = [kolon for kolon in train.columns if kolon not in ['id', 'Irrigation_Need']]

X = train[giris_sutunlari]
y = train['Irrigation_Need']
X_test = test[giris_sutunlari]


In [10]:
# 5-Fold Stratified Cross Validation ve Eğitim
# Sınıf dengesizliği olduğu için 'StratifiedKFold' kullanıyoruz
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_probabilities = np.zeros((len(train), 3))
test_probabilities = np.zeros((len(test), 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Çoklu sınıflandırma için LightGBM kuruyoruz
    model = LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        objective='multiclass',
        num_class=3,
        random_state=42,
        verbose=-1
    )
    
    model.fit(X_train, y_train)
    
    # Olasılık tahminlerini toplama
    oof_probabilities[val_idx] = model.predict_proba(X_val)
    test_probabilities += model.predict_proba(X_test) / skf.n_splits
    
    # Katman skoru hesaplama
    val_preds = np.argmax(oof_probabilities[val_idx], axis=1)
    fold_score = balanced_accuracy_score(y_val, val_preds)
    print(f"Katman {fold + 1} Balanced Accuracy Skoru: {fold_score:.4f}")

# Genel Başarı Oranı
oof_preds_final = np.argmax(oof_probabilities, axis=1)
genel_skor = balanced_accuracy_score(y, oof_preds_final)
print(f"\n---> TÜM VERİ SETİ GENEL BALANCED ACCURACY SKORU: {genel_skor:.4f} <---")


Katman 1 Balanced Accuracy Skoru: 0.9619
Katman 2 Balanced Accuracy Skoru: 0.9632
Katman 3 Balanced Accuracy Skoru: 0.9625
Katman 4 Balanced Accuracy Skoru: 0.9613
Katman 5 Balanced Accuracy Skoru: 0.9617

---> TÜM VERİ SETİ GENEL BALANCED ACCURACY SKORU: 0.9621 <---


In [11]:
# Kaggle Gönderi Dosyasını Hazırlama
# Tahmin edilen 0,1,2 sayılarını tekrar 'Low', 'Medium', 'High' kelimelerine çeviriyoruz
test_preds_final = np.argmax(test_probabilities, axis=1)
test_preds_labels = [inv_target_mapper[pred] for pred in test_preds_final]

submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': test_preds_labels
})


In [12]:
submission.to_csv('submission_irrigation.csv', index=False)